In [1]:
import numpy as np
import xarray as xr
from engin_core.convention import stamp, validate_timeseries

rng = np.random.default_rng(0)
n_run, n_time = 3, 12

ds = xr.Dataset(
    data_vars={
        "titer":   (("run", "time"), rng.random((n_run, n_time)) * 40, {"units": "g/L"}),
        "biomass": (("run", "time"), rng.random((n_run, n_time)) * 20, {"units": "g/L"}),
        "mu":      (("run", "time"), rng.random((n_run, n_time)) * 0.35, {"units": "1/h"}),
    },
    coords={
        "run":  ("run", [f"R{i:02d}" for i in range(n_run)]),
        "time": ("time", np.arange(float(n_time)), {"units": "h"}),
    },
)

report = validate_timeseries(stamp(ds))
print(report.summary())

conforms to engin convention 0.2: 0 error(s), 0 warning(s), 0 note(s)


In [2]:
broken = ds.copy()
del broken["titer"].attrs["units"]
broken["mu"].attrs["units"] = "per hour-ish"
broken["fluorescence"] = (("run", "time"), np.zeros((n_run, n_time)), {"units": "1"})

report = validate_timeseries(broken)
print(report.summary())
for finding in report.findings:
    print(f"  {finding}")

does not conform to engin convention 0.2: 2 error(s), 1 warning(s), 1 note(s)
  [warning] <dataset>: no 'engin_convention' attribute, so a reader cannot tell which version of the convention this was written under -- set ds.attrs["engin_convention"] = "0.2"
  [error] titer: no units attribute (this convention expects "g/L") -- set ds["titer"].attrs["units"] = "g/L"
  [error] mu: units 'per hour-ish' could not be parsed (TypeError) -- use a pint-parseable string, e.g. 'g/L', '1/h', 'degC'; dimensionless quantities are '1'
  [info] fluorescence: 'fluorescence' is not a registered channel, so it is carried without interpretation -- fine as-is; register it if it is common enough that other datasets will use the same name


In [3]:
import pandas as pd
from engin_core.convention import validate_endpoints

doe = pd.DataFrame({
    "run_id":    ["R00", "R01", "R02"],
    "feed_rate": [0.1, 0.2, 0.3],
    "titer":     [30.0, 35.0, 28.0],
})

report = validate_endpoints(doe, units={"titer": "g/L", "feed_rate": "L/h"})
print(report.summary())

conforms to engin convention 0.2: 0 error(s), 0 warning(s), 0 note(s)


In [4]:
from engin_core.convention import CHANNELS

for channel in CHANNELS.values():
    print(f"  {channel.name:14s} {channel.units:6s}  {channel.description}")

  titer          g/L     product concentration in broth
  biomass        g/L     dry cell weight
  substrate      g/L     limiting substrate concentration
  volume         L       working volume
  feed_rate      L/h     feed addition rate
  mu             1/h     specific growth rate
  do             %       dissolved oxygen, percent of saturation
  ph             1       pH, dimensionless by convention
  temperature    degC    broth temperature
  agitation      rpm     impeller speed
  airflow        vvm     gas flow per volume per minute
  offgas_co2     %       exhaust CO2 fraction
  offgas_o2      %       exhaust O2 fraction
  our            mmol/L/h  oxygen uptake rate
  cer            mmol/L/h  carbon dioxide evolution rate
  rq             1       respiratory quotient, CER/OUR
  kla            1/h     volumetric oxygen mass-transfer coefficient


In [5]:
from engin_core.convention import ROLES, DEFAULT_ROLE

for name, description in ROLES.items():
    print(f"  {name:10s} {description}")
print(f"\n  absent means {DEFAULT_ROLE!r}")

  measured   a sensor reading or a value derived from one -- the process value (PV)
  setpoint   the value the controller was asked to hold (SP)
  output     what the controller did about it -- an actuator command (OP/Out)

  absent means 'measured'


In [6]:
import pandas as pd
from engin_core.loaders import load_endpoints

messy = pd.DataFrame({
    "Batch":        ["B1", "B2"],
    "Titre (g/L)":  [30.0, 31.5],
    "OD600":        [4.1, 4.4],
    "O2 (%)":       [21.0, 20.8],
    "AUX2_raw":     [0.11, 0.09],
})

tidy, report = load_endpoints(messy)
print(report.summary())
print()
for guess in report.guesses:
    mapped = guess.channel or "—"
    print(f"  {guess.source:14s} -> {mapped:10s} {guess.confidence:>4}  {guess.evidence}")

endpoint table: 3 column(s) mapped, 1 unmapped, 1 below the 0.7 review threshold

  Titre (g/L)    -> titer      0.95  header matches a known alias for 'titer'; units 'g/L' agree with the convention
  OD600          -> biomass     0.9  header matches a known alias for 'biomass'; no units in the header, so the convention's default is assumed
  O2 (%)         -> do         0.45  header is ambiguous between ['do', 'offgas_o2']; took the first; units '%' agree with the convention
  AUX2_raw       -> —           0.0  no alias matched


In [7]:
for guess in report.needs_review:
    print(f"  check {guess.source!r}: {guess.evidence}")
    if guess.alternatives:
        print(f"    could also be: {guess.alternatives}")

  check 'O2 (%)': header is ambiguous between ['do', 'offgas_o2']; took the first; units '%' agree with the convention
    could also be: ['offgas_o2']


In [8]:
from engin_core.loaders import load_timeseries
from engin_core.convention import validate_timeseries

rows = [
    {"Batch": run, "Time (h)": float(t), "Titer (g/L)": 10.0 + t, "OD600": 2.0 + t}
    for run in ("R00", "R01")
    for t in range(4)
]

ds, load_report = load_timeseries(pd.DataFrame(rows))
print(load_report.summary())
print(validate_timeseries(ds).summary())
print(ds)

long table: 2 column(s) mapped, 0 unmapped, 0 below the 0.7 review threshold
conforms to engin convention 0.2: 0 error(s), 0 warning(s), 0 note(s)
<xarray.Dataset> Size: 176B
Dimensions:  (run: 2, time: 4)
Coordinates:
  * run      (run) object 16B 'R00' 'R01'
  * time     (time) float64 32B 0.0 1.0 2.0 3.0
Data variables:
    titer    (run, time) float64 64B 10.0 11.0 12.0 13.0 10.0 11.0 12.0 13.0
    biomass  (run, time) float64 64B 2.0 3.0 4.0 5.0 2.0 3.0 4.0 5.0
Attributes:
    engin_convention:  0.2


In [9]:
from engin_core.loaders import infer_columns, register_alias

register_alias("titer", "prod_a")
print(infer_columns(pd.DataFrame({"PROD_A": [1.0]})).guesses[0].evidence)

header matches a known alias for 'titer'; no units in the header, so the convention's default is assumed
